In [ ]:
#use "../lib/nb.ml";;
load_css ()

# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using the
scanner generator `ocamllex` and the parser generator `Menhir`.  (The Python version of this notebook uses `Ply`.)

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int' ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'for' '(' IF '=' expr '..' expr ')'
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: boolExpr ('&&' | '||') boolExpr
        | '!' boolExpr             
        | '(' boolExpr ')'
        | expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
This grammar is ambiguous.  We will later resolve the ambiguity by specifying the appropriate operator precedences. 

## Specification of the Scanner

`Ply` collects the definitions of the scanner from all variables and functions whose names start with `t_`.  The
scanner generator `ocamllex` reads a *scanner specification* instead, which is a text that consists of a *header*
containing OCaml code and a rule `token` with one case per token.  Every case consists of a regular expression and an
*action*, i.e. OCaml code that returns the token.  In order to stay close to the `Ply` version, we define the parts of
the scanner specification in variables whose names start with `t_` and assemble the specification afterwards.

In [ ]:
load_lib "parser_generator.ml";;
open Parse_tree

In `Ply`, the list `tokens` contains the names of the tokens.  In OCaml, the tokens are the values of a *variant
type* `token`, which we define in the module `Tokens`.  Both the scanner and the parser use this type.  Every token
carries a value of type `Parse_tree.tok`, which stores the type of the token as a string, the string that has been
recognized, its line, its column, and its position.  Hence a token knows the same things as a `LexToken` of `Ply`.

The tokens below the comment are the *literals* of `Ply`, see below.  `Ply` uses the character itself as the type of
such a token.  As a variant type needs names, we invent names for them.  The function `make_token` creates the token
that corresponds to the type of a value of type `tok`.

**Type safety:** `Ply` represents the type of a token as a string, so a misspelled token type is only detected when the
grammar is processed or, worse, when the token occurs in the input.  In OCaml, `make_token` is the only place where
strings are mapped to tokens; everywhere else the compiler checks the names of the tokens.

In [ ]:
module Tokens = struct
  type token =
    | NUMBER of tok | ID of tok | EQ of tok | NE of tok | LE of tok | GE of tok | AND of tok | OR of tok
    | INT of tok | IF of tok | ELSE of tok | WHILE of tok | FOR of tok | DOTDOT of tok | RETURN of tok
    (* the literals *)
    | PLUS of tok | MINUS of tok | TIMES of tok | DIVIDE of tok | MODULO of tok | LPAREN of tok | RPAREN of tok
    | LBRACE of tok | RBRACE of tok | SEMI of tok | ASSIGN of tok | LESS of tok | GREATER of tok | NOT of tok
    | COMMA of tok
    | EOF
end

open Tokens

let make_token (t : tok) : token =
  match t.kind with
  | "NUMBER" -> NUMBER t | "ID" -> ID t | "EQ" -> EQ t | "NE" -> NE t | "LE" -> LE t | "GE" -> GE t
  | "AND" -> AND t | "OR" -> OR t | "INT" -> INT t | "IF" -> IF t | "ELSE" -> ELSE t | "WHILE" -> WHILE t
  | "FOR" -> FOR t | "DOTDOT" -> DOTDOT t | "RETURN" -> RETURN t
  | "+" -> PLUS t | "-" -> MINUS t | "*" -> TIMES t | "/" -> DIVIDE t | "%" -> MODULO t | "(" -> LPAREN t
  | ")" -> RPAREN t | "{" -> LBRACE t | "}" -> RBRACE t | ";" -> SEMI t | "=" -> ASSIGN t | "<" -> LESS t
  | ">" -> GREATER t | "!" -> NOT t | "," -> COMMA t
  | kind -> invalid_arg ("make_token: unknown token type " ^ kind)

The token `NUMBER` specifies a natural number.  In `ocamllex`, characters are enclosed in single quotes and character
sets in brackets.

In [ ]:
let t_NUMBER = {|'0' | ['1'-'9'] ['0'-'9']*|}

Below, we define the tokens for operator symbols consisting of more than one character.  In `ocamllex`, a string can be
used as a regular expression, so there is no need to escape the character `|` as in `r'\|\|'`.

In [ ]:
let t_EQ     = {|"=="|}
let t_NE     = {|"!="|}
let t_LE     = {|"<="|}
let t_GE     = {|">="|}
let t_AND    = {|"&&"|}
let t_OR     = {|"||"|}
let t_DOTDOT = {|".."|}

Our version of `C` allows both *single-line comments* and *multi-line comments*.
- The regular expression `"/*" ([^ '*'] | '*'+ [^ '*' '/'])* '*'+ '/'` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.  The Python version uses the regular
  expression `/\*(.|\n)*?\*/` with the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a = 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment.  `ocamllex` has no non-greedy quantors.
  Therefore, our regular expression states explicitly that the comment contains no `*/` before its end.
- The regular expression `"//" [^ '\n']*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

A function like `t_COMMENT` of `Ply` becomes a pair consisting of a regular expression and an action.  The action
`token lexbuf` calls the scanner recursively and hence returns no token for the comment.  As a multi-line comment
contains newlines, the function `Parse_tree.new_lines` updates the line number of the scanner, like
`t.lexer.lineno += t.value.count('\n')`.

In [ ]:
let t_COMMENT = ({|"/*" ([^ '*'] | '*'+ [^ '*' '/'])* '*'+ '/' | "//" [^ '\n']*|},
                 {|Parse_tree.new_lines lexbuf; token lexbuf|})

The keywords `'int', 'if', 'else', 'while', 'for', 'return'` have to be dealt with separately as they are syntactical
identical to identifiers.  The association list `keywords` shown below maps every keyword to its *token type*.

In [ ]:
let keywords : (string * string) list =
  [ ("int"   , "INT");
    ("if"    , "IF");
    ("else"  , "ELSE");
    ("while" , "WHILE");
    ("for"   , "FOR");
    ("return", "RETURN")
  ]

When an identifier is read, we first have to check whether the identifier is one of our keywords.  If so, we assign the
corresponding token type that is stored in the list `keywords`.  Otherwise, the token type is set to `ID`.  The action
can use the variable `keywords`, since the generated scanner is loaded into the same toplevel.

In [ ]:
let t_ID = ({|['a'-'z' 'A'-'Z'] ['a'-'z' 'A'-'Z' '0'-'9' '_']*|},
            {|let t = Parse_tree.token "ID" lexbuf in
              make_token { t with kind = Option.value (List.assoc_opt t.value keywords) ~default:"ID" }|})

Operators consisting of a single character do not need an associated token type in `Ply`: their type is the character
itself.  The function `make_token` maps these types to the tokens `PLUS`, `MINUS`, and so on.

In [ ]:
let literals = [ '+'; '-'; '*'; '/'; '%'; '('; ')'; '{'; '}'; ';'; '='; '<'; '>'; '!'; ',' ]

White space, i.e. *space characters*, *tabulators*, and *carriage returns* are ignored. 

In [ ]:
let t_ignore = " \t\r"

Syntactically, newline characters are ignored. However, we still need to keep track of them in order to know the current line number, which is used for error messages.

In [ ]:
let t_newline = ({|'\n'|}, {|Lexing.new_line lexbuf; token lexbuf|})

Given the buffer `lexbuf` of the scanner, the function `find_column` returns the column where the token that has just
been scanned starts.  This is possible, because the scanner keeps track of the position of the beginning of the
current line in the field `pos_bol` (*position of the beginning of the line*), since the action of `t_newline` calls
`Lexing.new_line`.  Furthermore, `pos_cnum` is the number of characters that precede the token.

In [ ]:
let find_column (lexbuf : Lexing.lexbuf) : int =
  let p = Lexing.lexeme_start_p lexbuf in
  p.pos_cnum - p.pos_bol + 1

The function `t_error` is called for any character that can not be scanned by the lexer.  In this case,
`Lexing.lexeme lexbuf` is the character that is not recognized by the scanner.  This character is discarded.  After
that, scanning proceeeds as if nothing has happened:  the action that calls `t_error` calls the scanner recursively.

In [ ]:
let t_error (lexbuf : Lexing.lexbuf) : unit =
  let column = find_column lexbuf in
  Printf.printf "Illegal character '%s' in line %d, column %d.\n%!" (Lexing.lexeme lexbuf)
    (Lexing.lexeme_start_p lexbuf).pos_lnum column

The function `lex.lex` of `Ply` collects the definitions given above and builds the scanner.  In OCaml, we have to
assemble the scanner specification ourselves.  `Ply` adds the rules in the following order:  first the functions in
the order of their definition, then the strings sorted by the length of their regular expressions, the longest first.
`ocamllex` always prefers the *longest* match and, only if two rules match the same string, the rule that is given
first.  Hence, the order matters only for the rule `_`, which matches any character and therefore has to be the last
rule.  The function `generate_lexer` writes the specification to the file `_build/c_lexer.mll`, runs `ocamllex`,
and loads the generated module `C_lexer`.  The header `open Tokens` makes the tokens of the module `Tokens` available.

In [ ]:
let lexer : string =
  let function_rule (regexp, action) = Printf.sprintf "  | %s\n      { %s }" regexp action in
  let string_rule regexp kind = Printf.sprintf "  | %s { make_token (Parse_tree.token %S lexbuf) }" regexp kind in
  let char_set chars = "[" ^ String.concat " " (List.map (Printf.sprintf "%C") chars) ^ "]" in
  String.concat "\n"
    [ "{\n  open Tokens\n}\n";
      "rule token = parse";
      function_rule t_COMMENT;
      function_rule t_ID;
      function_rule t_newline;
      string_rule t_NUMBER "NUMBER";
      string_rule t_DOTDOT "DOTDOT";
      string_rule t_AND "AND";
      string_rule t_EQ "EQ";
      string_rule t_GE "GE";
      string_rule t_LE "LE";
      string_rule t_NE "NE";
      string_rule t_OR "OR";
      Printf.sprintf "  | %s { token lexbuf }" (char_set (List.of_seq (String.to_seq t_ignore)));
      Printf.sprintf "  | %s { make_token (Parse_tree.token (Lexing.lexeme lexbuf) lexbuf) }" (char_set literals);
      "  | eof { EOF }";
      "  | _ { t_error lexbuf; token lexbuf }" ]
;;
print_endline lexer;;
generate_lexer ~name:"c" ~lexer

In [ ]:
let test_scanner (file_name : string) : tok list =
  let program = read_file file_name in
  print_endline program;
  let lexbuf = Lexing.from_string program in   (* a new buffer starts with line number 1 *)
  let rec loop () =                            (* start scanning and collect all tokens *)
    match C_lexer.token lexbuf with
    | EOF -> []
    | NUMBER t | ID t | EQ t | NE t | LE t | GE t | AND t | OR t | INT t | IF t | ELSE t | WHILE t | FOR t
    | DOTDOT t | RETURN t | PLUS t | MINUS t | TIMES t | DIVIDE t | MODULO t | LPAREN t | RPAREN t | LBRACE t
    | RBRACE t | SEMI t | ASSIGN t | LESS t | GREATER t | NOT t | COMMA t ->
        t :: loop ()
  in
  loop ()

The function `lex_token_repr` shows a token like `Ply` shows a `LexToken`:  its type, its value, its line, and its
position.

In [ ]:
let lex_token_repr (t : tok) : string = Printf.sprintf "LexToken(%s,%s,%d,%d)" t.kind (python_repr t.value) t.line t.pos

In [ ]:
List.iter (fun t -> print_endline (lex_token_repr t)) (test_scanner "Examples/PrimesFor.c")

## Specification of the Parser

Like `Ply`, `Menhir` has no direct support for the *Ebnf* notation used above (apart from a library of parameterized
rules, which we do not use here).  Therefore, we rewrite the grammar that was given previously as follows:
```
program
    : function
    | function program

function 
    : INT ID '(' param_list ')' '{' decl_list stmnt_list '}'

param_list 
    : /* lambda */
    | INT ID 
    | INT ID ',' ne_param_list

ne_param_list
    : INT ID
    | INT ID ',' ne_param_list
    
decl_list
    : /* lambda */
    | INT ID ';' decl_list

stmnt_list
    : /* lambda */
    | stmnt stmnt_list
    
stmnt 
    : IF '(' bool_expr ')' stmnt                 
    | IF '(' bool_expr ')' stmnt ELSE stmnt
    | WHILE '(' bool_expr ')' stmnt
    | FOR '(' ID '=' expr DOTDOT expr ')'
    | '{' stmnt_list '}' 
    | ID '=' expr ';'  
    | RETURN expr ';'   
    | expr ';'       

bool_expr 
    : bool_expr '||' bool_expr
    | bool_expr '&&' bool_expr 
    | '!' bool_expr   
    | '(' bool_expr ')'
    | expr '==' expr     
    | expr '!=' expr     
    | expr '<=' expr     
    | expr '>=' expr     
    | expr '<'  expr      
    | expr '>'  expr     
 
expr: expr '+'  expr                 
    | expr '-' expr               
    | expr '*' expr               
    | expr '/' expr 
    | expr '%' expr 
    | '(' expr ')' 
    | NUMBER
    | ID                      
    | ID '(' expr_list ')'       

expr_list
    : /* lambda */
    | expr
    | expr ',' ne_expr_list

ne_expr_list
    : expr
    | expr ',' ne_expr_list

```
We will use precedence declarations to resolve the ambiguity that is inherent in this grammar.

Every grammar rule is followed by a *semantic action*, i.e. OCaml code in braces that computes the abstract syntax
tree of the left hand side from the abstract syntax trees of the symbols on the right hand side.  This corresponds to
the assignments to `p[0]` in the functions `p_program_one`, `p_function`, and so on of the `Ply` version.  Instead of
`p[1]`, `p[2]`, ..., the action uses names that are attached to the symbols, e.g. `f = function_`.

The Python version represents abstract syntax trees as *nested tuples*:
- a list of *n* items is represented as `('.', item₁, ⋯, itemₙ)`,
- a function definition is represented as `('fct', name, parameters, variables, statements)`,
- a variable is represented as a string, while a number is represented as `('Number', digits)`,
- all other nodes are tuples whose first component is the operator.

In OCaml, we define a type for every kind of abstract syntax tree instead:  `expr` for arithmetical expressions,
`bool_expr` for Boolean expressions, `stmnt` for statements, `fct` for function definitions, and `program` for
programs.  A list of *n* items is an OCaml list and a function definition is a record.  These types are defined
before the grammar, so that the actions can use them.

**Type safety:** The Python functions that generate code have to check the shape of every nested tuple at runtime.  In
OCaml, the types guarantee that, e.g., the condition of a `while` loop is a Boolean expression and the arguments of a
function call are arithmetical expressions.  Since `Menhir` knows the type of every syntactical variable, the
compiler also checks that every action computes a value of the right type.

In [ ]:
type arith_op = Add | Sub | Mul | Div | Mod
type cmp_op   = Eq | Ne | Le | Ge | Lt | Gt

type expr =
  | Var    of string                      (* a variable *)
  | Number of string                      (* the digits of a number *)
  | Arith  of arith_op * expr * expr
  | Call   of string * expr list          (* a function call *)

type bool_expr =
  | Compare of cmp_op * expr * expr
  | And     of bool_expr * bool_expr
  | Or      of bool_expr * bool_expr
  | Not     of bool_expr

type stmnt =
  | Block      of stmnt list
  | Assign     of string * expr
  | If         of bool_expr * stmnt
  | If_else    of bool_expr * stmnt * stmnt
  | While      of bool_expr * stmnt
  | For        of string * expr * expr * stmnt   (* for (var = lower .. upper) body *)
  | Return     of expr
  | Expr_stmnt of expr

type fct = { name : string; parameters : string list; variables : string list; stmnts : stmnt list }

type program = fct list

The *start variable* of our grammar is `program`.  `Menhir` needs a start rule that is followed by the end of the
input `EOF`.  Therefore, the start variable is `start`.  The declaration `%start` also states the type of the result of
the parser.

In [ ]:
let start   = {|%start <program> start|}
let p_start = {|start: p = program EOF { p }|}

Below, the precedence declarations for the tokens `'IF'` and `'ELSE'` are needed to solve the [dangling else problem](https://en.wikipedia.org/wiki/Dangling_else). 
The point is that the precedence of the keyword `'ELSE'` has to be higher than the precedence of the keyword `'if'` so that a program of the form
```
if (a == 1) if (b == 1) x = 2; else x = 3;
```
is interpreted as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } else {
        x = 3
    }
}
```
and not as 
```
if (a == 1) {
    if (a == 2) {
        x = 2;
    } 
} else {
    x = 3
}
```

In [ ]:
let precedence = {|
%nonassoc IF
%nonassoc ELSE
%left     OR
%left     AND
%right    "!"
%nonassoc EQ NE LE GE "<" ">"
%left     "+" "-"
%left     "*" "/" "%"
|}

The grammar rules are given as strings, too.  `Menhir` allows to declare an *alias* for every token.  We use the
aliases `"+"`, `"-"`, ..., which are declared below, for the literals, so that the rules look like the rules of `Ply`.
`function` is a keyword of OCaml and can not be used as the name of a syntactical variable, so we use `function_`.

In [ ]:
let p_program = {|
program: f = function_                            { [f] }
       | f = function_ p = program                { f :: p }
|}

In [ ]:
let p_function = {|
function_: INT n = ID "(" ps = param_list ")" "{" ds = decl_list ss = stmnt_list "}"
                                                  { { name = n.value; parameters = ps; variables = ds; stmnts = ss } }
|}

In [ ]:
let p_param_list = {|
param_list: /* empty */                           { [] }
          | INT n = ID                            { [n.value] }
          | INT n = ID "," ps = ne_param_list     { n.value :: ps }

ne_param_list: INT n = ID                         { [n.value] }
             | INT n = ID "," ps = ne_param_list  { n.value :: ps }
|}

In [ ]:
let p_decl_list = {|
decl_list: /* empty */                            { [] }
         | INT n = ID ";" ds = decl_list          { n.value :: ds }
|}

In [ ]:
let p_stmnt_list = {|
stmnt_list: /* empty */                           { [] }
          | s = stmnt ss = stmnt_list             { s :: ss }
|}

In the grammar rule
$$ \texttt{stmnt} \rightarrow \texttt{'if'}\;\texttt{'('}\; \texttt{bool_expr}\; \texttt{')'}\; \texttt{stmnt}$$
the rightmost token is `")"`.  However, this token does not have a precedence.  Therefore, the grammar rule
does not have a precedence either.  Hence, we manually assign the precedence of the token `IF` to this rule via 
the keyword `%prec`.  This way, the shift/reduce conflict resulting from the *dangling-else ambiguity* can be resolved.

In [ ]:
let p_stmnt = {|
stmnt: IF "(" b = bool_expr ")" s = stmnt %prec IF
                                                  { If (b, s) }
     | IF "(" b = bool_expr ")" s1 = stmnt ELSE s2 = stmnt
                                                  { If_else (b, s1, s2) }
     | WHILE "(" b = bool_expr ")" s = stmnt      { While (b, s) }
     | FOR "(" v = ID "=" l = expr DOTDOT u = expr ")" s = stmnt
                                                  { For (v.value, l, u, s) }
     | "{" ss = stmnt_list "}"                    { Block ss }
     | v = ID "=" e = expr ";"                    { Assign (v.value, e) }
     | RETURN e = expr ";"                        { Return e }
     | e = expr ";"                               { Expr_stmnt e }
|}

In [ ]:
let p_bool_expr = {|
bool_expr: l = bool_expr OR  r = bool_expr        { Or (l, r) }
         | l = bool_expr AND r = bool_expr        { And (l, r) }
         | "!" b = bool_expr                      { Not b }
         | "(" b = bool_expr ")"                  { b }
         | l = expr EQ  r = expr                  { Compare (Eq, l, r) }
         | l = expr NE  r = expr                  { Compare (Ne, l, r) }
         | l = expr LE  r = expr                  { Compare (Le, l, r) }
         | l = expr GE  r = expr                  { Compare (Ge, l, r) }
         | l = expr "<" r = expr                  { Compare (Lt, l, r) }
         | l = expr ">" r = expr                  { Compare (Gt, l, r) }
|}

In [ ]:
let p_expr = {|
expr: l = expr "+" r = expr                       { Arith (Add, l, r) }
    | l = expr "-" r = expr                       { Arith (Sub, l, r) }
    | l = expr "*" r = expr                       { Arith (Mul, l, r) }
    | l = expr "/" r = expr                       { Arith (Div, l, r) }
    | l = expr "%" r = expr                       { Arith (Mod, l, r) }
    | "(" e = expr ")"                            { e }
    | n = NUMBER                                  { Number n.value }
    | v = ID                                      { Var v.value }
    | f = ID "(" es = expr_list ")"               { Call (f.value, es) }
|}

In [ ]:
let p_expr_list = {|
expr_list: /* empty */                            { [] }
         | e = expr                               { [e] }
         | e = expr "," es = ne_expr_list         { e :: es }

ne_expr_list: e = expr                            { [e] }
            | e = expr "," es = ne_expr_list      { e :: es }
|}

The function `p_error` is called when the parser detects a syntax error.  `Menhir` reports a syntax error by raising
the exception `C_parser.Error`.  The function `yacc_parse` below catches this exception and calls `p_error` with the
buffer of the scanner, which contains the offending token.  At the end of the input, this token is the empty string.

In [ ]:
let p_error (lexbuf : Lexing.lexbuf) : unit =
  let column = find_column lexbuf in
  match Lexing.lexeme lexbuf with
  | "" -> print_endline "Syntax error at end of input."
  | value -> Printf.printf "Syntax error at token \"%s\" in line %d, column %d.\n%!" value (Lexing.lexeme_start_p lexbuf).pos_lnum column

The function `yacc.yacc` of `Ply` collects the rules from the functions given above and builds the parser.  In OCaml,
we assemble the grammar ourselves:  it starts with the declarations of the tokens, the start variable, the
precedences, and the types of the syntactical variables, which are separated from the grammar rules by `%%`.  The
tokens of the module `Tokens` are declared again, this time together with their aliases.  The option
`--external-tokens Tokens` tells `Menhir` that the tokens are defined by the module `Tokens` instead of the generated
module `C_parser`.  If the declarations did not agree with the module `Tokens`, the generated parser would not
type check.

The function `generate_grammar` writes the grammar to the file `_build/c_parser.mly`, runs `menhir`, and loads the
generated module `C_parser`.  The function `yacc_parse` corresponds to the function `yacc.parse` of `Ply`:  it parses a
string and returns `Some ast` or, after a syntax error, `None`.

In [ ]:
let grammar : string =
  String.concat "\n"
    [ {|%token <Parse_tree.tok> NUMBER ID EQ NE LE GE AND OR INT IF ELSE WHILE FOR DOTDOT RETURN|};
      {|%token <Parse_tree.tok> PLUS "+" MINUS "-" TIMES "*" DIVIDE "/" MODULO "%" LPAREN "(" RPAREN ")"|};
      {|%token <Parse_tree.tok> LBRACE "{" RBRACE "}" SEMI ";" ASSIGN "=" LESS "<" GREATER ">" NOT "!" COMMA ","|};
      {|%token EOF|};
      start;
      precedence;
      {|%type <program> program|};
      {|%type <fct> function_|};
      {|%type <string list> param_list ne_param_list decl_list|};
      {|%type <stmnt list> stmnt_list|};
      {|%type <stmnt> stmnt|};
      {|%type <bool_expr> bool_expr|};
      {|%type <expr> expr|};
      {|%type <expr list> expr_list ne_expr_list|};
      "%%";
      p_start; p_program; p_function; p_param_list; p_decl_list; p_stmnt_list; p_stmnt; p_bool_expr; p_expr; p_expr_list ]
;;
generate_grammar ~options:"--lalr --external-tokens Tokens" ~name:"c" ~grammar;;

let yacc_parse (program : string) : program option =
  let lexbuf = Lexing.from_string program in
  try Some (C_parser.start C_lexer.token lexbuf) with C_parser.Error -> p_error lexbuf; None

As we have used *precedence declarations* to resolve all shift/reduce conflicts, `Menhir` reports no conflict.
`Menhir` warns, however, that the precedence levels of the comparison operators are never used:  since a comparison
compares two arithmetical expressions and the result is a Boolean expression, the grammar can not be ambiguous with
respect to the comparison operators.  `Ply` keeps silent about such superfluous declarations.

Because of the option `--dump-resolved`, `Menhir` writes the states of the automaton into the file
`_build/c_parser.automaton.resolved`.  This file corresponds to the file `parser.out` of `Ply`.  As this file is long,
we show it with `display_text`.  The shift/reduce conflict resulting from the *dangling else problem* manifests itself
in state `94`:  the follow set of the marked rule
```
stmnt -> IF LPAREN bool_expr RPAREN stmnt . [ ... ELSE ]
```
contains the token `ELSE`, but there is also the marked rule `stmnt -> IF LPAREN bool_expr RPAREN stmnt . ELSE stmnt`.
The conflict is resolved in favor of shifting:  for the token `ELSE` the state offers only the transition to state `95`,
while `ELSE` is missing from the list of the tokens that cause a reduction.

In [ ]:
display_text (read_file "_build/c_parser.automaton.resolved")

The notebook `AST2Dot.ipynb` provides the function `tuple2dot`.  This function can be used to visualize the abstract
syntax tree that is generated by the function `yacc_parse`.

In [ ]:
run_notebook "../AST2Dot.ipynb"

The function `program_to_nested` converts an abstract syntax tree into a nested tuple of the type `nested` from the
notebook `AST2Dot.ipynb`.  This is the nested tuple that the Python version computes.  The function `nested_to_string`
from `AST2Dot.ipynb` writes it in the syntax of Python.

In [ ]:
let rec expr_to_nested (e : expr) : nested =
  let arith_op op = match op with Add -> "+" | Sub -> "-" | Mul -> "*" | Div -> "/" | Mod -> "%" in
  match e with
  | Var x -> Str x
  | Number n -> Tuple ("Number", [ Str n ])
  | Arith (op, lhs, rhs) -> Tuple (arith_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | Call (f, args) -> Tuple ("call", Str f :: List.map expr_to_nested args)

let rec bool_to_nested (b : bool_expr) : nested =
  let cmp_op op = match op with Eq -> "==" | Ne -> "!=" | Le -> "<=" | Ge -> ">=" | Lt -> "<" | Gt -> ">" in
  match b with
  | Compare (op, lhs, rhs) -> Tuple (cmp_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | And (lhs, rhs) -> Tuple ("&&", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Or (lhs, rhs) -> Tuple ("||", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Not arg -> Tuple ("!", [ bool_to_nested arg ])

let rec stmnt_to_nested (s : stmnt) : nested =
  match s with
  | Block stmnts -> Tuple (".", List.map stmnt_to_nested stmnts)
  | Assign (x, e) -> Tuple ("=", [ Str x; expr_to_nested e ])
  | If (b, s) -> Tuple ("if", [ bool_to_nested b; stmnt_to_nested s ])
  | If_else (b, s1, s2) -> Tuple ("if-else", [ bool_to_nested b; stmnt_to_nested s1; stmnt_to_nested s2 ])
  | While (b, s) -> Tuple ("while", [ bool_to_nested b; stmnt_to_nested s ])
  | For (x, lower, upper, s) -> Tuple ("for", [ Str x; expr_to_nested lower; expr_to_nested upper; stmnt_to_nested s ])
  | Return e -> Tuple ("return", [ expr_to_nested e ])
  | Expr_stmnt e -> expr_to_nested e

let fct_to_nested (f : fct) : nested =
  let names l = Tuple (".", List.map (fun x -> Str x) l) in
  Tuple ("fct", [ Str f.name; names f.parameters; names f.variables; Tuple (".", List.map stmnt_to_nested f.stmnts) ])

let program_to_nested (p : program) : nested = Tuple (".", List.map fct_to_nested p)

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed.
The resulting abstract syntax tree is printed as a nested tuple and visualized using `graphviz`.  Unlike the `Ply`
version, there is no need to reset the line number of the scanner:  `yacc_parse` creates a new buffer for every
string, and the line number of a new buffer is 1.

In [ ]:
let parse (file_name : string) : unit =
  let program = read_file file_name in
  print_endline program;
  match yacc_parse program with
  | None -> print_endline "None"
  | Some ast ->
      print_endline (nested_to_string (program_to_nested ast));
      Graphviz.display (tuple2dot (program_to_nested ast))

In [ ]:
parse "Examples/PrimesFor.c"

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
let indent (s : string) : string = String.make 8 ' ' ^ s

The function `compile_expr expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents an expression.  
  This abstract syntax tree has the type `expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** The Python function needs a final case `case _: assert False` for nested tuples that are not
expressions.  As the type `expr` contains only expressions, the OCaml compiler checks that every case is handled, and
the final case is not needed.  For the same reason, the Python dictionary `OpToCmd` becomes a `match` on the operator.

In [ ]:
let rec compile_expr (expr : expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Var var -> (* expr is a variable name *)
      let cmd = indent (Printf.sprintf "iload %d" (Dict.find var st)) in
      ([ cmd ], 1)
  | Number n ->
      let cmd = indent ("ldc " ^ n) in
      ([ cmd ], 1)
  | Arith (op, lhs, rhs) ->
      let l1, sz1 = compile_expr lhs st class_name in
      let l2, sz2 = compile_expr rhs st class_name in
      let op_to_cmd = match op with Add -> "iadd" | Sub -> "isub" | Mul -> "imul" | Div -> "idiv" | Mod -> "irem" in
      let cmd = indent op_to_cmd in
      (l1 @ l2 @ [ cmd ], max sz1 (1 + sz2))
  | Call ("println", args) ->
      let cmd_lst   = ref [ indent "getstatic java/lang/System/out Ljava/io/PrintStream;" ] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + 1 + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokevirtual java/io/PrintStream/println(%s)V" (String.make !cnt 'I')) ];
      (!cmd_lst, !stck_size)
  | Call (f, args) ->
      let cmd_lst   = ref [] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokestatic %s/%s(%s)I" class_name f (String.make !cnt 'I')) ];
      (!cmd_lst, max !stck_size 1)

The following is a test of the function `compile_expr`.

In [ ]:
let expr = Call ("sum", [ Arith (Add, Var "x", Var "y") ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let expr = Arith (Add, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

The variable `label_counter` is a global counter that is used to create unique label names.
Every call of `new_label` creates a new, unique label.

In [ ]:
let expr = Call ("println", [ Var "x"; Var "y" ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let label_counter = ref 0

let new_label () : string =
  label_counter := !label_counter + 1;
  "l" ^ string_of_int !label_counter

The function `compile_bool expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents a Boolean expression.  
  This abstract syntax tree has the type `bool_expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** In Python, Boolean expressions and arithmetical expressions are both nested tuples, so nothing prevents
us from calling `compile_bool` with an arithmetical expression.  In OCaml, they have the different types `bool_expr`
and `expr`, and the type checker rejects such a call.

In [ ]:
let rec compile_bool (expr : bool_expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Compare (op, lhs, rhs) ->
      let op_to_cmd =
        match op with
        | Eq -> "if_icmpeq"
        | Ne -> "if_icmpne"
        | Le -> "if_icmple"
        | Ge -> "if_icmpge"
        | Lt -> "if_icmplt"
        | Gt -> "if_icmpgt"
      in
      let l1, sz1    = compile_expr lhs st class_name in
      let l2, sz2    = compile_expr rhs st class_name in
      let true_label = new_label () in
      let next_label = new_label () in
      let cmd_lst    = l1 @ l2 in
      let cmd        = op_to_cmd in
      let cmd_lst    = cmd_lst @ [ indent (cmd ^ " " ^ true_label) ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 0" ] in
      let cmd_lst    = cmd_lst @ [ indent ("goto " ^ next_label) ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ true_label ^ ":" ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 1" ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ next_label ^ ":" ] in
      (cmd_lst, max sz1 (1 + sz2))
  | And (lhs, rhs) | Or (lhs, rhs) ->
      let op_to_cmd = match expr with And _ -> "iand" | _ -> "ior" in
      let l1, sz1   = compile_bool lhs st class_name in
      let l2, sz2   = compile_bool rhs st class_name in
      let cmd       = op_to_cmd in
      let cmd_lst   = l1 @ l2 @ [ indent cmd ] in
      (cmd_lst, max sz1 (1 + sz2))
  | Not arg ->
      let l, sz   = compile_bool arg st class_name in
      let cmd_lst = [ indent "bipush 1" ] @ l @ [ indent "isub" ] in
      (cmd_lst, sz + 1)

Below is a test for the function `compile_bool`.

In [ ]:
let expr = Compare (Eq, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Not (Compare (Eq, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Or (Compare (Eq, Var "x", Var "y"), Compare (Lt, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

The function `compile_stmnt stmnt st class_name` takes three arguments:
- `stmnt` is an *abstract syntax tree* that represents a statement.  
  This abstract syntax tree has the type `stmnt`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

The Python version recognizes an expression statement by the fact that no other case matches.  In OCaml, an expression
statement is a value of the form `Expr_stmnt expr`.

In [ ]:
let rec compile_stmnt (stmnt : stmnt) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match stmnt with
  | Assign (var, expr) ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent (Printf.sprintf "istore %d" (Dict.find var st)) ] in
      (cmd_lst, sz)
  | If (expr, sub_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt sub_stmnt st class_name in
      let else_label = new_label () in
      let lbl_stmnt  = String.make 4 ' ' ^ else_label ^ ":" in
      let cmd_lst    = l1 @ [ indent ("ifeq " ^ else_label) ] @ l2 @ [ lbl_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | If_else (expr, then_stmnt, else_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt then_stmnt st class_name in
      let l3, sz3    = compile_stmnt else_stmnt st class_name in
      let else_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ else_label) in
      let else_stmnt = String.make 4 ' ' ^ else_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ next_label) in
      let cmd_lst    = l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; else_stmnt ] @ l3 @ [ next_stmnt ] in
      (cmd_lst, max sz1 (max sz2 sz3))
  | While (expr, body_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt body_stmnt st class_name in
      let loop_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ next_label) in
      let loop_stmnt = String.make 4 ' ' ^ loop_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ loop_label) in
      let cmd_lst    = [ loop_stmnt ] @ l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; next_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | For (var, lower, upper, body_stmnt) ->
      let l1, sz1      = compile_expr lower st class_name in
      let l2, sz2      = compile_expr upper st class_name in
      let l3, sz3      = compile_stmnt body_stmnt st class_name in
      let loop_label   = new_label () in
      let next_label   = new_label () in
      let loop_stmnt   = String.make 4 ' ' ^ loop_label ^ ":" in
      let next_stmnt   = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt   = indent ("goto " ^ loop_label) in
      let istore_stmnt = indent (Printf.sprintf "istore %d" (Dict.find var st)) in
      let iload_stmnt  = indent (Printf.sprintf "iload  %d" (Dict.find var st)) in
      let iinc_stmnt   = indent (Printf.sprintf "iinc %d 1" (Dict.find var st)) in
      let cmp_stmnt    = indent ("if_icmpgt " ^ next_label) in
      let cmd_lst =
        l1 @ [ istore_stmnt; loop_stmnt; iload_stmnt ] @ l2 @ [ cmp_stmnt ] @ l3 @ [ iinc_stmnt; goto_stmnt; next_stmnt ]
      in
      (cmd_lst, max (max 2 sz1) (max (sz2 + 1) sz3))
  | Return expr ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent "ireturn" ] in
      (cmd_lst, sz)
  | Block stmnt_lst ->
      let cmd_lst = ref [] in
      let size    = ref 0 in
      List.iter
        (fun s ->
          let l, sz = compile_stmnt s st class_name in
          cmd_lst := !cmd_lst @ l;
          size    := max !size sz)
        stmnt_lst;
      (!cmd_lst, !size)
  | Expr_stmnt expr -> (* it is an expression statement *)
      compile_expr expr st class_name

In [ ]:
let stmnt = Assign ("x", Number "42")
let st    = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If (Compare (Eq, Var "x", Var "y"), Assign ("x", Number "42"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If_else (Compare (Lt, Var "x", Var "y"), Assign ("x", Var "y"), Assign ("y", Var "x"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = While (Compare (Lt, Var "x", Var "y"), Assign ("x", Arith (Add, Var "x", Number "1")));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = For ("i", Number "1", Var "n", Assign ("s", Arith (Add, Var "s", Var "i")))
let st    = Dict.of_list [ ("i", 0); ("n", 1); ("s", 2) ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let st    = Dict.of_list [ ("x", 0); ("y", 1) ]
let stmnt = Block [ Assign ("x", Var "y"); Assign ("x", Number "1"); Assign ("y", Var "x") ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let compile_fct (fct_def : fct) (class_name : string) : string list * int =
  label_counter := 0;
  let { name; parameters; variables; stmnts } = fct_def in
  let m   = List.length parameters in
  let n   = List.length variables in
  let st  = ref Dict.empty in
  let cnt = ref 0 in
  List.iter
    (fun var ->
      st  := Dict.add var !cnt !st;
      cnt := !cnt + 1)
    (parameters @ variables);
  let cmd_lst = ref [] in
  let size    = ref 0 in
  List.iter
    (fun stmnt ->
      let l, sz = compile_stmnt stmnt !st class_name in
      cmd_lst := !cmd_lst @ l;
      size    := max !size sz)
    stmnts;
  let limit_locals = Printf.sprintf ".limit locals %d" (m + n) in
  let limit_stack  = Printf.sprintf ".limit stack  %d" !size in
  let return_stmnt = indent "return" in
  if name <> "main" then begin
    let meth    = Printf.sprintf ".method public static %s(%s)I" name (String.make m 'I') in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ ".end method" ] in
    (cmd_lst, !size)
  end
  else begin
    let meth    = ".method public static main([Ljava/lang/String;)V" in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ return_stmnt; ".end method" ] in
    (cmd_lst, !size)
  end

In [ ]:
let f = { name = "sum"; parameters = [ "x" ]; variables = [ "y"; "z" ]; stmnts = [ Return (Var "x") ] };;
compile_fct f "Sum"

In [ ]:
(* Instead of the Python module os.path, we use the module Filename of the OCaml standard library. *)

In [ ]:
let file = "~/Drive/Kurse/Formal-Languages/Python/Chapter-13/Examples/Primes.c";;
print_endline (Filename.dirname file);;
print_endline (Filename.basename file);;
print_endline (Filename.concat "abc" "xyz.c")

The input to `compile_program(file_name)` is the name of a `C` file.  This file assumed to have the ending `.c`.
This file is compiled and the resulting assembler file is written into a file with the same name but the ending `.jas`.

In [ ]:
let compile_program (file_name : string) : unit =
  let directory = Filename.dirname file_name in
  let base      = Filename.basename file_name in
  let base      = Filename.chop_suffix base ".c" in
  let outfile   = Filename.concat directory (base ^ ".jas") in
  let program   = read_file file_name in
  match yacc_parse program with
  | None -> ()   (* the syntax error has already been reported *)
  | Some fct_lst ->
      let cmd_lst = ref [] in
      List.iter
        (fun fct ->
          let l, _ = compile_fct fct base in
          cmd_lst := !cmd_lst @ l @ [ "\n" ])
        fct_lst;
      Out_channel.with_open_text outfile (fun handle ->
          output_string handle (".class public " ^ base ^ "\n");
          output_string handle ".super java/lang/Object\n\n";
          output_string handle ".method public <init>()V\n";
          output_string handle "    aload 0\n";
          output_string handle "    invokenonvirtual java/lang/Object/<init>()V\n";
          output_string handle "    return\n";
          output_string handle ".end method\n\n";
          List.iter (fun cmd -> output_string handle (cmd ^ "\n")) !cmd_lst)

The file `Primes.c` is a simple `C` program that computes the prime numbers in a naive way.

In [ ]:
Sys.chdir "Examples";;
Sys.getcwd ()

In [ ]:
shell "ls -l"

In [ ]:
cat "PrimesFor.c"

In [ ]:
compile_program "PrimesFor.c"

In [ ]:
shell "ls -l"

In [ ]:
cat "PrimesFor.jas"

Next, we generate Java byte code using
[jasmin](http://jasmin.sourceforge.net).

In [ ]:
shell "jasmin PrimesFor.jas"

Finally, we run the generated byte code.

In [ ]:
shell "ls -l"

In [ ]:
shell "java PrimesFor"